# Sicurre — CamemBERTav2 Training

Thin orchestration wrapper. All logic lives in `src/`.

**Dataset sources required on Kaggle:**
- `sicurre-ml-src` — versioned copy of `src/` from this repo (uploaded by CI)
- `sicurre-data` — frozen training CSV splits (synced from Cloudflare R2)

**Kaggle Secrets required:**  
`HF_TOKEN`, `HF_USERNAME`, `REPO_NAME`, `DATABRICKS_HOST`, `DATABRICKS_TOKEN`,  
`DATABRICKS_EMAIL`, `MLFLOW_EXPERIMENT_NAME`

In [ ]:
# ── Cell 0: Environment setup ─────────────────────────────────────────
# Install any packages not pre-installed in the Kaggle GPU image.
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "accelerate>=1.1",
     "evaluate>=0.4",
     "mlflow-skinny>=3.0",
     "scipy>=1.10",
     "sentencepiece>=0.2",
    ],
    check=True,
)

# Make the repo src/ package available for import.
sys.path.insert(0, "/kaggle/input/sicurre-ml-src")

In [ ]:
# ── Cell 1: Imports ───────────────────────────────────────────────────
from pathlib import Path

import mlflow
import numpy as np

from src.config.training_config import (
    RuntimeState,
    TrainingConfig,
    detect_device,
    detect_runtime,
    load_secrets,
)
from src.data.loader import load_splits
from src.data.tokenizer import prepare_tokenized_splits
from src.evaluation import (
    build_error_dataframe,
    confusion_matrix_arrays,
    evaluate_on_test,
    save_classification_report,
)
from src.model.builder import compute_class_weights, load_model
from src.registry import promote_if_threshold, push_to_hub, register_model, setup_mlflow
from src.training.baseline import prepare_baseline_training

In [ ]:
# ── Cell 2: Runtime + secrets ─────────────────────────────────────────
runtime_env = detect_runtime()
device, use_tpu = detect_device()
secrets = load_secrets(runtime_env)

print(f"Runtime : {runtime_env.upper()}")
print(f"Device  : {device}  |  TPU: {use_tpu}")

In [ ]:
# ── Cell 3: Configuration ─────────────────────────────────────────────
from datetime import datetime

config = TrainingConfig()

DATA_DIR   = Path("/kaggle/input/sicurre-data")
OUTPUT_DIR = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

state = RuntimeState(
    runtime_env=runtime_env,
    device=device,
    use_tpu=use_tpu,
    run_date=datetime.now().strftime("%Y%m%d"),
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    secrets=secrets,
    hf_token=secrets.get("HF_TOKEN"),
    databricks_host=secrets.get("DATABRICKS_HOST"),
    databricks_token=secrets.get("DATABRICKS_TOKEN"),
    databricks_email=secrets.get("DATABRICKS_EMAIL"),
    mlflow_experiment_name=secrets.get("MLFLOW_EXPERIMENT_NAME") or "sicurre-phishing",
)

HF_USERNAME = secrets.get("HF_USERNAME", "")
REPO_NAME   = secrets.get("REPO_NAME", "sicurre-phishing-detector")
HF_REPO_ID  = f"{HF_USERNAME}/{REPO_NAME}"
MODEL_NAME  = "main.sicurre.phishing-detector"

print(f"Output  : {OUTPUT_DIR}")
print(f"HF repo : {HF_REPO_ID}")

In [ ]:
# ── Cell 4: MLflow setup ──────────────────────────────────────────────
experiment_path = setup_mlflow(state)
print(f"Experiment : {experiment_path}")
print(f"Tracking   : {mlflow.get_tracking_uri()}")

In [ ]:
# ── Cell 5: Load data ─────────────────────────────────────────────────
splits = load_splits(DATA_DIR, config.label2id)

print(f"Train : {len(splits.train):,}  |  Val : {len(splits.val):,}  |  Test : {len(splits.test):,}")

In [ ]:
# ── Cell 6: Tokenize ──────────────────────────────────────────────────
tok_splits = prepare_tokenized_splits(
    splits=splits,
    model_name=config.model_name,
    max_length=config.max_length,
    hf_token=state.hf_token,
)

print(f"Tokenized — train: {len(tok_splits.train)}  val: {len(tok_splits.val)}  test: {len(tok_splits.test)}")

In [ ]:
# ── Cell 7: Model + class weights ─────────────────────────────────────
model = load_model(config, hf_token=state.hf_token)

class_weights = compute_class_weights(
    splits.train["label"].values,
    num_labels=config.num_labels,
    strategy=config.class_weight_strategy,
)

print(f"Class weights: {class_weights.tolist()}")

In [ ]:
# ── Cell 8: Baseline setup ────────────────────────────────────────────
setup = prepare_baseline_training(
    train_df=splits.train,
    tokenized_splits=tok_splits,
    config=config,
    runtime=state,
)
trainer = setup.trainer

print(f"Run name    : {setup.run_name}")
print(f"Output dir  : {setup.output_dir}")
print(f"Warmup steps: {setup.training_args.warmup_steps}")

In [ ]:
# ── Cell 9: Train ─────────────────────────────────────────────────────
with mlflow.start_run(run_name=setup.run_name) as run:
    mlflow.log_params(setup.run_config)
    trainer.train()
    mlflow.log_metrics({k: v for k, v in trainer.state.log_history[-1].items() if isinstance(v, float)})

baseline_run_id = run.info.run_id
print(f"Run ID: {baseline_run_id}")

In [ ]:
# ── Cell 10: Evaluate on test set ─────────────────────────────────────
test_metrics = evaluate_on_test(trainer, tok_splits.test)

with mlflow.start_run(run_id=baseline_run_id):
    mlflow.log_metrics(test_metrics)
    mlflow.end_run()

print(f"F1 weighted     : {test_metrics.get('test_f1_weighted', 0):.4f}")
print(f"Phishing recall : {test_metrics.get('test_phishing_recall', 0):.4f}")

In [ ]:
# ── Cell 11: Save model ───────────────────────────────────────────────
save_path = setup.output_dir / "best"
trainer.save_model(str(save_path))
tok_splits.tokenizer.save_pretrained(str(save_path))
print(f"Saved to {save_path}")

In [ ]:
# ── Cell 12: Error analysis + confusion matrix ────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
from src.config.training_config import LABEL_NAMES

test_preds = trainer.predict(tok_splits.test)
pred_labels = np.argmax(test_preds.predictions, axis=-1)
true_labels = splits.test["label"].values

error_df = build_error_dataframe(splits.test, test_preds.predictions)
errors   = error_df[~error_df["correct"]]
print(f"Errors: {len(errors)} / {len(error_df)} ({100*len(errors)/len(error_df):.1f}%)")

cm, cm_norm = confusion_matrix_arrays(true_labels, pred_labels)
report_path = save_classification_report(true_labels, pred_labels, OUTPUT_DIR)
print(f"Report saved: {report_path}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=axes[0])
axes[0].set_title("Confusion Matrix (counts)")
sns.heatmap(cm_norm, annot=True, fmt=".3f", cmap="Blues",
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=axes[1])
axes[1].set_title("Confusion Matrix (normalised)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# ── Cell 13: Register model in MLflow Unity Catalog ───────────────────
model_info = register_model(
    run_id=baseline_run_id,
    save_path=save_path,
    model_name=MODEL_NAME,
    config=config,
)
print(f"Registered: {MODEL_NAME} v{model_info.registered_model_version}")

In [ ]:
# ── Cell 14: Promote ──────────────────────────────────────────────────
promoted = promote_if_threshold(
    model_name=MODEL_NAME,
    test_metrics=test_metrics,
    recall_threshold=0.97,
    f1_threshold=0.90,
)
alias = "@production" if promoted else "@staging"
print(f"Alias assigned: {alias}")

In [ ]:
# ── Cell 15: Push to HuggingFace (only if @production) ────────────────
if promoted and state.hf_token:
    hf_url = push_to_hub(
        save_path=save_path,
        hf_repo_id=HF_REPO_ID,
        hf_token=state.hf_token,
        test_metrics=test_metrics,
        mlflow_version=str(model_info.registered_model_version),
        artifact_dir=OUTPUT_DIR,
    )
    print(f"Live: {hf_url}")
else:
    print("Skipped HF push — model did not reach @production threshold.")